# Example: E. coli Drug Time-Course Phenotyping

This notebook demonstrates how to analyze E. coli morphological changes under various drug treatments over time. It highlights path-based metadata extraction and exploratory analysis.

**Dataset Information:**
- **Source:** [E. coli CLSM Dataset](https://bbbc.broadinstitute.org/)
- **Description:** Confocal Laser Scanning Microscopy (CLSM) images of E. coli treated with different antibiotics.
- **Metadata:** Extracted from the folder structure: `.../(drug)_#(replicate)/(time)/(img_name)/(crop_name).*`.

**Workflow Overview:**
1. Discover files and extract metadata from paths.
2. Extract deep learning embeddings using DINOv2.
3. Compute morphological properties.
4. Analyze drug effects via clustering and distance metrics.
5. Explain results using embedding-property correlations.

In [ ]:
import torch
from phenome import PhenoMe, load_dinov2_model

# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize PhenoMe and load model
pheno = PhenoMe(device=device, seed=42)
wrapper = load_dinov2_model("dinov2_vitg14_reg", device=device)

## 1. Paths & Configuration

Specify the paths to your dataset. This example uses a folder structure where metadata is encoded in the path.

In [ ]:
DATA_DIR = "/datasets/Ecoli_CLSM/preprocessed_BB/"
MASK_DIR = "/datasets/Ecoli_CLSM/preprocessed_masks_BB/"

## 2. File Discovery & Metadata Extraction

We use `get_metadata_from_path` to automatically extract 'drug', 'replicate', and 'time' from the directory structure.

In [ ]:
from phenome.utils import get_metadata_from_path

# Define the path pattern to extract metadata
path_pattern = ".../(drug)_#(replicate)/(time)/(img_name)/(crop_name).*"

file_df = pheno.find_files(
    DATA_DIR,
    mask_dir=MASK_DIR,
    extensions=[".tif", ".tiff"],
    metadata_fn=get_metadata_from_path(path_pattern),
)

# Clean up metadata values
file_df["time"] = file_df["time"].replace({"0_min": "00_min", "2_min": "02_min", "5_min": "05_min"})
file_df["drug"] = file_df["drug"].str.replace(r"(?i)_replicate$", "", regex=True)
pheno.set_file_df(file_df)

print(f"Drugs found: {sorted(file_df['drug'].unique())}")
print("Available time options:", sorted(file_df["time"].unique()))
file_df.head()

## 3. Feature Extraction

Process images to get deep embeddings and compute morphological properties.

In [ ]:
eval_drugs = ["Control", "Chloramphenicol", "Azide", "MP265", "Mecillinam", "Nalidixate", "Rifampicin"]
eval_times = ["60_min"]

# Extract embeddings with filters
pheno.process_images(
    wrapper,
    filters={"drug": eval_drugs, "time": eval_times},
    batch_size=328,
    channel_mode="split",
    resize_size=224,  # Target size for resizing images
    pad_size=168,  # Pad small images to 168x168 before resizing to 224x224
)

In [ ]:
# Compute properties
df_properties = pheno.compute_properties(
    property_preset="full_extended",
    n_jobs=8,
)

## 4. Analysis: Clustering & Centroids

We visualize the centroids of different drug treatments.

In [ ]:
pheno.plot_centroids(
    group_by=["drug"],
    method="tsne",
    show_points=True,
)

## 5. Interactive Exploration

Explore the phenospace interactively.

In [ ]:
e = pheno.create_interactive_explorer()